# 1D network particle tracking on the Delaware River Basin

Passive tracer demo driven by pywatershed's network hydraulics export. The input file is produced by
`examples/02a_network_hydraulics_export.ipynb` in the pywatershed repository; set its path below.

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection

from fluvial_particle import (FileHydraulicsProvider, Network, NetworkConfig, estimate_particles,
                              run_network_simulation)

DRB = pathlib.Path("/home/rmcd/projects/pywatershed/examples/02a_network_hydraulics_export/drb_network_hydraulics.nc")
OUT = pathlib.Path("./network-drb-demo-output")
TRENTON = 4205

import textwrap

def finish(fig, name, caption):
    """Add a caption under the figure, save it to OUT as a 150 dpi PNG for sharing, and show it."""
    fig.text(0.5, -0.02, textwrap.fill(caption, 125), ha="center", va="top", fontsize=9.5, wrap=True)
    path = OUT / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path}")
    plt.show()


## 1. The network

The per-step dispersion-kick vs. reach-length diagnostic (the "dt check" line) is printed by
`run_network_simulation` when it runs, in section 3 below -- not here.

In [ ]:
prov = FileHydraulicsProvider(DRB)
net = Network(prov.static, crs_wkt=prov.crs_wkt)
print(f"{net.n_reach} reaches, {net.outlets().size} outlets, {net.headwaters().size} headwaters, "
      f"{net.length.sum()/1000:.0f} km; {prov.times[0]} .. {prov.times[-1]}")
last = prov.hydraulics(prov.times[-1])
fig, ax = plt.subplots(figsize=(7, 9))
lc = LineCollection([np.column_stack(p) for p in net.polylines()], array=last["velocity"], cmap="viridis", linewidths=1.2)
ax.add_collection(lc); ax.autoscale(); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(lc, label=f"reach velocity on {str(prov.times[-1])[:10]} (m/s)")
ax.set_title(f"Delaware River Basin network: {net.n_reach} reaches, {net.outlets().size} outlets, {net.headwaters().size} headwaters")
finish(fig, "fig1_drb_network", (
    f"Figure 1. The pywatershed PRMS segment network for the Delaware River Basin as exported to the network "
    f"hydraulics file: {net.n_reach} reaches totalling {net.length.sum() / 1000:.0f} km, coloured by the routed "
    f"velocity on the last day of the export ({str(prov.times[0])[:10]} to {str(prov.times[-1])[:10]}, daily). "
    f"Reach {TRENTON} at the bottom is the main outlet (Trenton)."))


## 2. Sources: a slug at every headwater plus a week-long loading on the mainstem

We first estimate a per-source particle mass with `estimate_particles` (targeting about 50 particles per
500 m bin), then adopt the finest suggested mass -- the minimum across sources -- so every source meets that
target; we fall back to the median instead if that minimum would push the total particle count past roughly
200,000.

In [ ]:
start = np.datetime64("1979-03-01")
heads = net.headwaters()
# Mainstem: walk upstream from Trenton following the largest-flow parent (reused in sections 5 and 7).
path_idx = [net.index_of(TRENTON)]
while True:
    parents = net.parents(path_idx[-1])
    if parents.size == 0:
        break
    path_idx.append(int(parents[np.argmax(last["flow_out"][parents])]))
path_idx = path_idx[::-1]  # upstream -> downstream
path_km = np.cumsum(net.length[path_idx]) / 1000.0
mainstem = net.id_of(path_idx[int(np.searchsorted(path_km, 0.4 * path_km[-1]))])  # ~40 % of the way down
print(f"mainstem: {len(path_idx)} reaches, {path_km[-1]:.0f} km from its headwater to Trenton; "
      f"loading source on reach {mainstem}, {path_km[path_idx.index(net.index_of(mainstem))]:.0f} km down")
sources = [{"reach_id": int(r), "form": "slug", "time": 0.0, "mass": 100.0} for r in heads]
sources.append({"reach_id": int(mainstem), "form": "loading", "rate": 0.05, "start": "1979-03-03", "end": "1979-03-10"})
cfg_dict = {
    "hydraulics_file": str(DRB), "start_time": str(start), "end_time": str(start + np.timedelta64(30, "D")),
    "dt": 900.0, "output_interval": 3600.0, "particle_mass": 1.0, "mass_units": "kg",
    "dispersion": {"model": "fischer"}, "sources": sources, "seed": 42,
}
budget = estimate_particles(NetworkConfig.from_dict(cfg_dict), prov, target_per_bin=50, bin_length=500.0)
print(budget)

total_mass = float(budget["total_mass"].sum())
particle_mass = float(budget["particle_mass"].min())
total_particles = total_mass / particle_mass
choice = "the finest suggested mass (min), so every source meets its target bin occupancy"
if total_particles > 200_000:
    particle_mass = float(budget["particle_mass"].median())
    total_particles = total_mass / particle_mass
    choice = "the median suggested mass (the min would exceed ~200k total particles)"
print(f"chosen particle_mass = {particle_mass:.6g} kg via {choice}")
print(f"total particles at chosen particle_mass = {total_particles:.0f}")

cfg = NetworkConfig.from_dict({**cfg_dict, "particle_mass": particle_mass})


## 3. Run one month

In [ ]:
res = run_network_simulation(cfg, OUT)
print(res.summary())


## 4. Arrival-time distributions at the outlets

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for outlet in net.outlets():
    h = res.arrival_histogram(int(outlet), bin_seconds=6 * 3600.0)
    if h.mass.sum() > 0:
        ax.plot(h.time, h.mass, label=f"reach {outlet}" + (" (Trenton)" if outlet == TRENTON else ""))
pos = res.positions(-1)
released = pos.loc[pos.status != 0, "mass"].sum()  # mass released by the final output time (status != 0)
recovered = res.arrival_times().mass.sum()
print(f"released {released:.0f} kg, recovered at outlets {recovered:.0f} kg ({100*recovered/released:.0f}%)")
ax.set_xlabel("date"); ax.set_ylabel("mass arriving per 6 h (kg)")
ax.set_title("Mass arriving at the basin outlets")
loading = [r for r in cfg.sources if r["form"] == "loading"][0]
ax.legend(title=f"{len(heads)} headwater slugs x 100 kg at {str(start)[:10]}; loading {float(loading['rate']):g} kg/s on reach {mainstem}")
finish(fig, "fig2_outlet_arrivals", (
    f"Figure 2. Breakthrough curves at the six outlets: particle mass arriving per 6 h bin. Sources are a 100 kg slug "
    f"at every headwater at {str(start)[:10]} plus a continuous loading of {float(loading['rate']):g} kg/s on mainstem reach {mainstem} from "
    f"{loading['start']} to {loading['end']}. Of {released:.0f} kg released, {recovered:.0f} kg ({100 * recovered / released:.0f}%) "
    f"reached an outlet within the {(np.datetime64(cfg.end_time, 'D') - np.datetime64(cfg.start_time, 'D')).astype(int)}-day run; "
    f"the remainder sits in a headwater reach with zero flow. Trenton (reach {TRENTON}) carries almost all of it."))


## 5. Concentration along the mainstem

In [ ]:
# path_idx (the mainstem, upstream -> downstream) comes from section 2
bins = res.bins(500.0)
sel = np.isin(bins.bin_reach, path_idx)
order = np.argsort([path_idx.index(r) for r in bins.bin_reach[sel]], kind="stable")
dist = np.cumsum(bins.bin_width[sel][order]) / 1000.0

raw = res.concentration(None, bin_length=500.0, smoothing=None).values[:, sel][:, order]
sm = res.concentration(None, bin_length=500.0, smoothing="auto").values[:, sel][:, order]
vmax = float(np.nanpercentile(raw, 99))

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
axes[0].pcolormesh(dist, res.times, raw, shading="nearest", cmap="magma", vmin=0, vmax=vmax)
axes[0].set_title("raw 500 m bins")
im1 = axes[1].pcolormesh(dist, res.times, sm, shading="nearest", cmap="magma", vmin=0, vmax=vmax)
axes[1].set_title("smoothed (auto bandwidth)")
for ax in axes:
    ax.set_xlabel("distance along mainstem (km)")
axes[0].set_ylabel("date")
last_hour = int(np.nonzero(np.nansum(raw, axis=1) > 0)[0][-1])   # clip the time axis a day past the last mass on the mainstem
axes[0].set_ylim(res.times[0], res.times[min(last_hour + 24, res.times.size - 1)])
fig.colorbar(im1, ax=axes, label="concentration (kg/m3)")
fig.suptitle(f"Concentration along the mainstem, {dist[-1]:.0f} km from its headwater to Trenton")
finish(fig, "fig3_mainstem_concentration", (
    f"Figure 3. Distance-time plot of concentration along the mainstem walked upstream from Trenton by largest-flow "
    f"parent ({len(path_idx)} reaches). Left: raw particle mass per 500 m bin divided by width x depth x 500 m from "
    f"the daily hydraulics; right: the same with a Gaussian kernel of bandwidth sqrt(2 K dt) per reach. The bright "
    f"band starting on 1979-03-03 is the continuous loading travelling downstream; the faint early streaks are the "
    f"headwater slugs. Colour scale clipped at the 99th percentile of the raw values."))


## 6. Map animation: particle positions and reach concentration

In [ ]:
reach_da = res.reach_concentration(None)
reach_cube = reach_da.values  # dims (time, bin), one bin per reach in reach-index order (matches net.polylines())
vmax_map = float(np.nanpercentile(reach_cube, 99))

# Stop the animation shortly after the last moving particle exits instead of playing an empty network.
# Particles released into a reach with zero flow wait there (the export's dry-reach convention), so the
# cutoff is where the active count first falls to the number still parked at the end of the run.
status = res.positions()["status"].values  # (time, particle): 0 unreleased, 1 active, 2 exited
active = (status == 1).sum(axis=1)
peak_idx = int(np.argmax(active))
parked = int(active[-1])
settled = np.nonzero(active[peak_idx:] <= parked)[0]
end_idx = peak_idx + int(settled[0]) if settled.size else res.times.size - 1
end_frame = min(res.times.size - 1, end_idx + 12)  # a 12-hour tail after the last exit
if parked:
    where = res.positions(-1).query("status == 1").reach_id.value_counts()
    print(f"{parked} particles never move: released into zero-flow reach(es) {where.index.tolist()}")
print(f"peak active particles {active[peak_idx]} at {str(res.times[peak_idx])[:16]}; "
      f"last moving particle exited by {str(res.times[end_idx])[:16]}; "
      f"animating {str(res.times[0])[:10]} .. {str(res.times[end_frame])[:16]}")

fig, (ax_p, ax_c) = plt.subplots(1, 2, figsize=(13, 9))
segments = [np.column_stack(p) for p in net.polylines()]

# left: particles on a gray network
ax_p.add_collection(LineCollection(segments, colors="lightgray", linewidths=0.8, zorder=0))
scat = ax_p.scatter([], [], s=4, c="crimson", zorder=2)
count_text = ax_p.set_title("")
ax_p.set_xlabel("particle positions")

# right: a thin gray network, with reaches drawn in color only where concentration > 0 and
# a line width that grows with concentration (thin at the low end, thick at the color scale's maximum)
ax_c.add_collection(LineCollection(segments, colors="lightgray", linewidths=0.8, zorder=0))
cmap = plt.get_cmap("viridis")
norm = matplotlib.colors.Normalize(0, vmax_map)
lc = LineCollection(segments, cmap=cmap, norm=norm, zorder=1)
ax_c.add_collection(lc)

def concentration_style(c):
    """Line widths for the colored layer: 0 (hidden) where c is NaN or 0, else 1 to 6 with c / vmax_map."""
    c = np.nan_to_num(c, nan=0.0)
    widths = np.where(c > 0.0, 1.0 + 5.0 * np.clip(c / vmax_map, 0.0, 1.0), 0.0)
    return np.ma.masked_invalid(c), widths

values, widths = concentration_style(reach_cube[0])
lc.set_array(values); lc.set_linewidths(widths)
fig.colorbar(lc, ax=ax_c, label=f"reach concentration ({reach_da.attrs['units']})", shrink=0.8)
ax_c.set_xlabel("reach concentration")

# source indicators and a caption describing the loading, all derived from the config
slug_rows = [r for r in cfg.sources if r["form"] == "slug"]
load_rows = [r for r in cfg.sources if r["form"] != "slug"]
def release_xy(rows):
    idx = np.array([net.index_of(int(r["reach_id"])) for r in rows], dtype=np.int64)
    return net.map_position(idx, np.zeros(idx.size))
sx, sy = release_xy(slug_rows)
lx, ly = release_xy(load_rows)
ax_p.scatter(sx, sy, s=22, facecolors="none", edgecolors="royalblue", linewidths=0.9, zorder=1,
             label=f"{len(slug_rows)} headwater slugs at t = 0")
for ax in (ax_p, ax_c):
    ax.scatter(lx, ly, marker="*", s=220, c="gold", edgecolors="black", linewidths=0.8, zorder=3,
               label="continuous loading")
ax_p.legend(loc="lower right", fontsize=9)

slug_mass = sum(float(r["mass"]) for r in slug_rows)
load = load_rows[0]
load_start, load_end = np.datetime64(load["start"]), np.datetime64(load["end"])
load_days = float((load_end - load_start) / np.timedelta64(1, "D"))
caption = (f"Slugs: {len(slug_rows)} headwaters x {float(slug_rows[0]['mass']):g} {cfg.mass_units} at t = 0 "
           f"({slug_mass:g} {cfg.mass_units} total).   "
           f"Loading: {float(load['rate']):g} {cfg.mass_units}/s on reach {load['reach_id']} "
           f"from {str(load_start)[:10]} to {str(load_end)[:10]} ({load_days:g} days, "
           f"{float(load['rate']) * load_days * 86400:g} {cfg.mass_units}).   "
           f"Particle mass {particle_mass:.3g} {cfg.mass_units}.")
fig.text(0.5, 0.04, caption, ha="center", va="center", fontsize=9.5)

for ax in (ax_p, ax_c):
    ax.autoscale(); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
suptitle = fig.suptitle("", y=0.97)
frames = range(0, end_frame + 1, 2)  # every second output hour

def update(i):
    df = res.map_positions(int(i))
    ok = df.status == 1
    scat.set_offsets(np.column_stack([df.x[ok], df.y[ok]]))
    count_text.set_text(f"active particles: {int(active[i])}")
    values, widths = concentration_style(reach_cube[i])
    lc.set_array(values); lc.set_linewidths(widths)
    loading_on = load_start <= res.times[i] < load_end
    suptitle.set_text(f"{str(res.times[i])[:16]}     loading {'ON' if loading_on else 'off'}")
    return scat, count_text, lc, suptitle

anim = FuncAnimation(fig, update, frames=frames, interval=200, blit=False)
anim.save(OUT / "drb_particles.gif", writer="pillow", fps=5)
plt.close(fig)
from IPython.display import Image
Image(filename=str(OUT / "drb_particles.gif"))

## 7. Concentration versus time at four sites

Four reaches chosen from the topology rather than by id, so they follow the sources if the setup changes:
the reach carrying the continuous loading, a mainstem reach midway between it and Trenton, the mainstem
reach where the largest tributary joins, and Trenton itself. Reach concentration has one bin per reach, so
these are the raw per-reach values.

In [ ]:
src_idx = net.index_of(int(mainstem))
tr_idx = net.index_of(TRENTON)
i_src, i_tr = path_idx.index(src_idx), path_idx.index(tr_idx)
mid_idx = path_idx[(i_src + i_tr) // 2]

# the largest tributary joining the mainstem between the loading source and Trenton
best_flow, trib_idx = 0.0, None
for r in path_idx[i_src : i_tr + 1]:
    for parent in net.parents(r):
        if parent not in path_idx and float(last["flow_out"][parent]) > best_flow:
            best_flow, trib_idx = float(last["flow_out"][parent]), int(parent)
conf_idx = int(net.to_index[trib_idx]) if trib_idx is not None else path_idx[i_src + 1]

sites = {
    "loading source": src_idx,
    "mid-mainstem": mid_idx,
    "largest tributary confluence": conf_idx,
    "Trenton (outlet)": tr_idx,
}
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
polylines = net.polylines()

fig, (ax_ts, ax_map) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [2, 1]})
for (name, idx), color in zip(sites.items(), colors):
    ax_ts.plot(res.times, reach_cube[:, idx], color=color, label=f"{name} (reach {net.id_of(idx)})")
    x, y = polylines[idx]
    ax_map.plot(x, y, color=color, linewidth=4, label=name)
ax_ts.set_xlim(res.times[0], res.times[min(res.times.size - 1, end_frame + 24)])
ax_ts.set_xlabel("time")
ax_ts.set_ylabel(f"concentration ({reach_da.attrs['units']})")
ax_ts.set_title("Reach concentration versus time at four sites")
ax_ts.legend(title="site (reach id)")
ax_map.add_collection(LineCollection([np.column_stack(p) for p in polylines], colors="lightgray", linewidths=0.8, zorder=0))
ax_map.autoscale(); ax_map.set_aspect("equal"); ax_map.set_title("site locations"); ax_map.set_xticks([]); ax_map.set_yticks([])
ax_map.legend(fontsize=8, loc="lower right")
finish(fig, "fig4_site_concentration", (
    f"Figure 4. Reach-average concentration versus time at four sites chosen from the topology: the reach carrying the "
    f"continuous loading (reach {net.id_of(src_idx)}), a mainstem reach midway to Trenton (reach {net.id_of(mid_idx)}), the "
    f"mainstem reach where the largest tributary joins (reach {net.id_of(conf_idx)}), and Trenton (reach {TRENTON}). "
    f"Each curve is the particle mass in the reach divided by its flow volume at that hour. The loading pulse arrives at "
    f"each site in downstream order and its shut-off on 1979-03-10 propagates the same way; the early bumps are headwater "
    f"slugs passing through. The map shows the four reaches in the same colours."))

In [ ]:
res.close(); prov.close()
